# Notebook 55 — Security Primitives

Demonstrates `multigen.security`: API key management, HS256 JWT (stdlib only),
workflow data classification, network policy enforcement, compliance scanning,
and the `SecurityManager` facade.  No real secrets or external services needed.

In [ ]:
import sys, importlib.util
sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

sec = load('security')
print('security loaded OK')

## 1. APIKeyManager — issue key, validate correct scope, raise InvalidAPIKeyError on wrong scope

In [ ]:
key_mgr = sec.APIKeyManager()

# Issue a key with read + write scopes, valid for 30 days
key = key_mgr.issue(owner='service-a', scopes=['read', 'write'], ttl_days=30)
print('Issued token (prefix):', key.token[:12] + '...')
print('Owner   :', key.owner)
print('Scopes  :', key.scopes)
print('is_valid:', key.is_valid)

# Validate with a required scope that the key has
validated = key_mgr.validate(key.token, required_scope='write')
print('\nValidate(write) — owner:', validated.owner)

# Validate with a scope the key does NOT have
try:
    key_mgr.validate(key.token, required_scope='admin')
except sec.InvalidAPIKeyError as exc:
    print('InvalidAPIKeyError (wrong scope):', exc)

# Revoke and re-validate
key_mgr.revoke(key.token)
try:
    key_mgr.validate(key.token)
except sec.InvalidAPIKeyError as exc:
    print('InvalidAPIKeyError (revoked)    :', exc)

## 2. JWTManager — sign claims, verify and decode, show JWTError on tampered token

In [ ]:
jwt_mgr = sec.JWTManager(secret='super-secret-key', ttl_seconds=3600)

# Sign a claims dict
claims = {'sub': 'alice', 'role': 'admin', 'org': 'TechCorp'}
token = jwt_mgr.sign(claims)
print('JWT (first 40 chars):', token[:40] + '...')
print('Token parts count:', len(token.split('.')))

# Verify and decode
decoded = jwt_mgr.verify(token)
print('\nDecoded claims:', decoded)
print('sub  :', decoded['sub'])
print('role :', decoded['role'])
print('iat present:', 'iat' in decoded)
print('exp present:', 'exp' in decoded)

# Tamper with the token — modify payload section
parts = token.split('.')
tampered_token = parts[0] + '.' + parts[1][:-2] + 'XX' + '.' + parts[2]
try:
    jwt_mgr.verify(tampered_token)
except sec.JWTError as exc:
    print('\nJWTError on tampered token:', exc)

## 3. WorkflowClassifier — 3 classification rules, classify 4 different contexts

In [ ]:
classifier = sec.WorkflowClassifier()
classifier.add_rule(sec.DataClassification.RESTRICTED,   ['ssn', 'credit_card', 'passport_number'])
classifier.add_rule(sec.DataClassification.CONFIDENTIAL, ['salary', 'medical_record', 'diagnosis'])
classifier.add_rule(sec.DataClassification.INTERNAL,     ['employee_id', 'client_id', 'internal'])

contexts = [
    {'query': 'What is the weather today?'},
    {'query': 'Show employee_id list for the team'},
    {'patient': 'John', 'diagnosis': 'hypertension', 'treatment': 'lisinopril'},
    {'user': 'alice', 'ssn': '123-45-6789', 'credit_card': '4111...'},
]

for ctx in contexts:
    level = classifier.classify(ctx)
    snippet = str(ctx)[:60]
    print(f'{level.value:12s} | {snippet}')

## 4. NetworkPolicy (default_deny=True) — allow OpenAI + Anthropic, block everything else

In [ ]:
policy = sec.NetworkPolicy(default_deny=True)
policy.allow(r'https://api\.openai\.com/.*')
policy.allow(r'https://api\.anthropic\.com/.*')

allowed_urls = [
    'https://api.openai.com/v1/chat/completions',
    'https://api.anthropic.com/v1/messages',
]
blocked_urls = [
    'https://evil.com/exfiltrate',
    'https://random-service.io/api',
]

print('Allowed URLs:')
for url in allowed_urls:
    result = policy.check(url)
    print(f'  OK: {url}')

print('\nBlocked URLs:')
for url in blocked_urls:
    try:
        policy.check(url)
    except sec.NetworkPolicyViolation as exc:
        print(f'  BLOCKED: {url}')
        print(f'    Reason: {exc}')

## 5. ComplianceScan — no_pii and no_profanity rules, clean passes, PII raises ComplianceViolation

In [ ]:
scan = sec.ComplianceScan()

scan.add_rule(sec.ComplianceRule(
    name='no_pii',
    description='No PII (email addresses) in workflow input',
    check_fn=lambda ctx: '@' not in str(ctx.get('input', '')),
    severity='error',
))
scan.add_rule(sec.ComplianceRule(
    name='no_profanity',
    description='No profanity in input',
    check_fn=lambda ctx: 'badword' not in str(ctx.get('input', '')).lower(),
    severity='error',
))

# Clean context — should pass
clean_ctx = {'input': 'Summarise the Q3 earnings report.', 'agent': 'finance-bot'}
report = scan.scan(clean_ctx)
print('Clean context — passed:', report.passed)
print('Violations:', report.violations)

# PII context — should raise ComplianceViolation
pii_ctx = {'input': 'Send report to alice@example.com immediately.', 'agent': 'mailer'}
try:
    scan.scan(pii_ctx)
except sec.ComplianceViolation as exc:
    print('\nComplianceViolation (PII):', exc)

## 6. SecurityManager facade — keys + JWT + classifier + compliance in one workflow guard

In [ ]:
manager = sec.SecurityManager(jwt_secret='prod-secret-12345')

# Add a compliance rule
manager.compliance.add_rule(sec.ComplianceRule(
    name='no_pii',
    description='No email addresses in workflow input',
    check_fn=lambda ctx: '@' not in str(ctx.get('input', '')),
    severity='error',
))

# Add a network allow rule
manager.network.allow(r'https://api\.openai\.com/.*')

def workflow_guard(api_token: str, jwt_token: str, ctx: dict, url: str) -> dict:
    """
    Guard function that checks: API key, JWT, data classification,
    network policy, and compliance before a workflow is allowed to run.
    Returns a summary dict.
    """
    result = {}

    # 1. Validate API key
    key = manager.keys.validate(api_token, required_scope='run')
    result['api_key_owner'] = key.owner

    # 2. Verify JWT
    claims = manager.jwt.verify(jwt_token)
    result['jwt_sub'] = claims['sub']

    # 3. Classify data
    level = manager.classifier.classify(ctx)
    result['data_classification'] = level.value

    # 4. Network policy
    manager.network.check(url)
    result['url_allowed'] = True

    # 5. Compliance scan
    report = manager.compliance.scan(ctx)
    result['compliance_passed'] = report.passed

    return result

# Wire up
api_key   = manager.keys.issue('workflow-runner', scopes=['run', 'read'])
jwt_token = manager.jwt.sign({'sub': 'alice', 'role': 'analyst'})

safe_ctx  = {'input': 'Analyse Q3 earnings data.', 'agent': 'finance-bot'}
safe_url  = 'https://api.openai.com/v1/chat/completions'

summary = workflow_guard(api_key.token, jwt_token, safe_ctx, safe_url)
print('Workflow guard summary (safe workflow):')
for k, v in summary.items():
    print(f'  {k}: {v}')

# Unsafe: PII in input raises ComplianceViolation before workflow starts
unsafe_ctx = {'input': 'Email results to ceo@company.com', 'agent': 'mailer'}
try:
    workflow_guard(api_key.token, jwt_token, unsafe_ctx, safe_url)
except sec.ComplianceViolation as exc:
    print('\nBlocked by compliance scan (PII in input):', exc)